This tutorial can be used to upload your scPoli model to scvi-hub

In [ ]:
!pip install scvi
!pip install scarches

In [ ]:
import scarches as sca
import torch
from scvi.model.base._constants import SAVE_KEYS
import os
import scvi

In [1]:
!pip install huggingface-hub==0.16.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 4.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.27.0
    Uninstalling huggingface-hub-0.27.0:
      Successfully uninstalled huggingface-hub-0.27.0


In [2]:
from huggingface_hub import login, ModelCard
card = ModelCard.load("scvi-tools/tabula-sapiens-blood-scanvi")

/Users/chelseaalexandra.bright/miniconda3/envs/scarches2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
card

In [41]:
card.text[6322:6822]

'<summary><strong>Summary Statistics</strong></summary>\n\n|     Summary Stat Key     | Value |\n|--------------------------|-------|\n|         n_batch          |   6   |\n|         n_cells          | 34766 |\n| n_extra_categorical_covs |   0   |\n| n_extra_continuous_covs  |   0   |\n|         n_labels         |  19   |\n|       n_latent_qzm       |  20   |\n|       n_latent_qzv       |  20   |\n|          n_vars          | 3000  |\n\n</details>\n\n\n<details>\n<summary><strong>Training</strong></summary>\n\n<!--'

In [48]:
# pattern = r"\|\s*(.*?)\s*\|\s*(.*?)\s*\|"

# pattern = r"Summary Stat Key     \| Value \|\\n\|--------------------------\|-------\|\\n\|         n_batch          \|   6   \|\\n\|         n_cells          \| 34766 \|\\n\| n_extra_categorical_covs \|   0   \|\\n\| n_extra_continuous_covs  \|   0   \|\\n\|         n_labels         \|  19   \|\\n\|       n_latent_qzm       \|  20   \|\\n\|       n_latent_qzv       \|  20   \|\\n\|          n_vars          \| 3000  \|"
pattern = r"n_batch([\s\S]*?)n_cells"
pattern1 = r"Training"


match = re.search(pattern, card.text)
match1 = re.search(pattern1, card.text)
print(match)
print(match1)

<re.Match object; span=(6462, 6506), match='n_batch          |   6   |\n|         n_cells'>
<re.Match object; span=(1897, 1905), match='Training'>


In [47]:
match.span()

(6462, 6506)

In [ ]:

import re, json
import os
def parse_json_from_text(text):
    """
    Parses the "model_setup_anndata_args" object from a given text block and returns it as a Python dictionary.
    
    Args:
        text (str): The input text containing the JSON object.
        
    Returns:
        dict or None: If the JSON object is successfully parsed, returns a Python dictionary.
                      If parsing fails or if the JSON object is not found, returns None.
    """
    # Define the pattern to find the "model_setup_anndata_args" JSON object
    pattern = r"\*\*model_setup_anndata_args\*\*:\s*```json\s*(.*?)```"

    # Find the JSON object using regex
    match = re.search(pattern, text, re.DOTALL)

    if match:
        json_str = match.group(1)
        
        # Parse the JSON object
        try:
            json_object = json.loads(json_str)
            return json_object
        except json.JSONDecodeError as e:
            print("Error decoding JSON:", e)
            return None
    else:
        print("JSON object not found in the text.")
        return None  
    
model_setup_anndata_args = parse_json_from_text(card.text)

JSON object not found in the text.


Below, please specify the directory where your scPoli model files (attr.pkl, model_params.pt, and var_names.csv) are saved. By default the directory is set to the current working directory.

In [ ]:
# load scPoli model
attr_dict, model_state_dict, var_names = sca.models.scPoli._load_params("classifiers/data/model_hnoca_scpoli", map_location="cpu")
model = sca.models.scPoli.load("classifiers/data/model_hnoca_scpoli", map_location="cpu")

The cells below rearrange the existing information in the attr_dict of the scPoli model in order to be of a correct format to create an scvi-hub Model Card

In [ ]:
init_params_ = model._get_init_params_from_dict(attr_dict)
setup_keys = ["condition_keys","cell_type_keys"]
setup_args = {k: v for k, v in init_params_.items() if k in setup_keys}
keys_to_drop=["condition_keys","cell_type_keys","share_metadata","obs_metadata","prototypes_labeled","cov","conditions","cell_types","conditions_combined","labeled_indices"]
init_params = {k: v for k, v in init_params_.items() if k not in keys_to_drop}
attr_dict["init_params_"]=init_params

In [ ]:
attr_dict["registry_"]={}
attr_dict["registry_"]["scvi_version"]="n/a"
attr_dict["registry_"]["model_name"]="SCPOLI"
attr_dict["registry_"]["setup_args"]=setup_args
attr_dict["registry_"]["field_registries"]={}
attr_dict["registry_"]["field_registries"]["summary_stats"]={}
attr_dict["registry_"]["field_registries"]["data_registry"]={}

In [ ]:
# add summary stats
attr_dict["registry_"]["field_registries"]={'X':{}, 'batch':{}, 'labels':{}, 'size_factor':{}, 'extra_categorical_covs':{}, 'extra_continuous_covs':{}}
attr_dict["registry_"]["field_registries"]["X"]["summary_stats"]={'n_vars': model.adata.n_vars, 'n_cells': model.adata.n_obs}
attr_dict["registry_"]["field_registries"]["batch"]["summary_stats"]={'n_batch': len(attr_dict["conditions_combined_"])}
attr_dict["registry_"]["field_registries"]["labels"]["summary_stats"]={'n_labels': len(attr_dict["cell_types_"].keys())}
attr_dict["registry_"]["field_registries"]["size_factor"]["summary_stats"]={}
attr_dict["registry_"]["field_registries"]["extra_categorical_covs"]["summary_stats"]={}
attr_dict["registry_"]["field_registries"]["extra_continuous_covs"]["summary_stats"]={}

In [ ]:
# add data registry
for registry_key, field_registry in attr_dict["registry_"]["field_registries"].items():
    field_registry["data_registry"] = {}

The cell below converts your scPoli model files into one single .pt file output. Below you can optionally set a specific file output path and prefix. By default the output model will be saved in the current working directory under the name "model.pt"

In [ ]:
# specify output path and output file prefix (default to current working directory with no prefix)
dir_path = "classifiers/data/model_hnoca_scpoli"
file_name_prefix = ""
save_kwargs={}

model_save_path = os.path.join(dir_path, f"{file_name_prefix}{SAVE_KEYS.MODEL_FNAME}")

# only save the public attributes with _ at the very end
user_attributes = {k: v for k, v in attr_dict.items() if k[-1] == "_"}


torch.save(

{

SAVE_KEYS.MODEL_STATE_DICT_KEY: model_state_dict,

SAVE_KEYS.VAR_NAMES_KEY: var_names,

SAVE_KEYS.ATTR_DICT_KEY: user_attributes,

},

model_save_path,

**save_kwargs,

)

In what follows, we read in our scPoli .pt model and create an scvi-hub model card

In [ ]:
!pip install huggingface_hub

In [ ]:
from scvi.hub import HubMetadata, HubModel, HubModelCardHelper
import anndata
model_path = "classifiers/data/model_hnoca_scpoli"

hm = HubMetadata.from_dir(model_path, anndata_version=anndata.__version__, map_location='cpu')


In [ ]:
hmch = HubModelCardHelper.from_dir(
    model_path,
    license_info="cc-by-4.0",
    anndata_version=anndata.__version__,
    data_modalities=["rna"],
    data_is_annotated=True,
    description="HNOCA atlas",
    model_parent_module="scarches.model",
    data_is_minified=True,
    references="He, Z., Dony, L., Fleck, J.S. et al. An integrated transcriptomic cell atlas of human neural organoids. Nature 635, 690–698 (2024). https://doi.org/10.1038/s41586-024-08172-8",
)

You can save your model card to a markdown file and make changes on disk

In [ ]:
hmch.model_card.save(
    "my_model_card.md"
)  # then change the markdown file on disk...

In [ ]:
print(hmch.model_card.content)

Create HubModel and upload it

In [ ]:
hmo = HubModel(model_path, metadata=hm, model_card=hmch)
hmo

Your model card can now be pushed to the scvi-hub. For more information on how to make an access token, see this [article](https://huggingface.co/docs/hub/security-tokens). For more details on uploading a model to scvi-hub, see this [tutorial](https://docs.scvi-tools.org/en/1.0.0/tutorials/notebooks/scvi_hub_upload_and_large_files.html)

In [ ]:
access_token="" #please specify your access token that you can create on your huggingface profile
hmo.push_to_huggingface_hub(
    repo_name="scvi-tools/human-neural-organoid-cell-atlas-scpoli", repo_token=access_token, repo_create=True
)